# TW-CA-kNN: Testability-Weighted Co-Association kNN-50

Run all cells. When asked, upload the four CSVs from the repository's `data/` folder (or one zip containing them): `structural_features.csv`, `testability_features.csv`, `net_edges.csv`, `net_driver_cells.csv`.

## 1. Settings

In [1]:
RUNS = 20                       # runs per circuit
N_JOBS = -1                     # parallel workers (-1 = all cores)
USE_GITHUB = False              # True: fetch data/ from the GitHub repository instead of uploading
REPO_URL = 'https://github.com/reethveekO-o/tw-caknn-trojan-detection.git'

## 2. Setup and data

In [2]:
import importlib, subprocess, sys
if importlib.util.find_spec('networkx') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'networkx'], check=True)

import os, glob, zipfile, time, random, warnings
from collections import defaultdict
import numpy as np, pandas as pd, networkx as nx
from networkx.algorithms.community import modularity
from scipy.stats import rankdata
from sklearn.cluster import KMeans
from sklearn.metrics import roc_auc_score, average_precision_score
from joblib import Parallel, delayed
from IPython.display import display
warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)
IN_COLAB = 'google.colab' in sys.modules

In [3]:
REQUIRED = ['structural_features.csv', 'testability_features.csv', 'net_edges.csv', 'net_driver_cells.csv']

def find_data_dir():
    cands = ['.', 'data', 'tw-caknn-trojan-detection/data', '/content', '/content/data',
             '/content/tw-caknn-trojan-detection/data'] + glob.glob('**/', recursive=True)[:200]
    for d in cands:
        if all(os.path.exists(os.path.join(d, f)) for f in REQUIRED):
            return d
    return None

DATA_DIR = find_data_dir()
if DATA_DIR is None and USE_GITHUB:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL], check=True)
    DATA_DIR = find_data_dir()
if DATA_DIR is None and IN_COLAB:
    from google.colab import files
    print('Upload the four CSVs (or one zip containing them):')
    up = files.upload()
    os.makedirs('/content/data', exist_ok=True)
    for name, blob in up.items():
        if name.lower().endswith('.zip'):
            with open('/tmp/up.zip', 'wb') as fh: fh.write(blob)
            zipfile.ZipFile('/tmp/up.zip').extractall('/content/data')
        else:
            with open(os.path.join('/content/data', name), 'wb') as fh: fh.write(blob)
    DATA_DIR = find_data_dir()
if DATA_DIR is None:
    raise FileNotFoundError('Data not found. Put the four CSVs next to this notebook or in data/, '
                            'or set USE_GITHUB = True. Needed: ' + ', '.join(REQUIRED))
print('Data folder:', os.path.abspath(DATA_DIR))

Data folder: /home/claude/nb/data


## 3. TW-CA-kNN

In [4]:
POOL, MES, K_NN, SEED = 100, 20, 50, 1           # pool size, members per run, k, master seed
SMALL_N = 1000                                   # Louvain branch below, 1-hop branch at or above
BETA, SEED_FRAC, POOL_FRAC, CA_MAJ = 2.0, 0.7, 0.5, 0.5
XGB22 = ['f0', 'f1', 'f2', 'f3', 'f4', 'f28', 'f29', 'f34', 'f45', 'f46', 'f48',
         'f50', 'f52', 'f53', 'f54', 'f55', 'CC0', 'CC1', 'CO', 'SC0', 'SC1', 'SO']

# clustering ensemble, co-association, kNN-50 rarity
def zscore(X):
    s = X.std(0, ddof=1); s[s == 0] = 1
    return (X - X.mean(0)) / s

def kmeans_pool(Z, rng, size=POOL):
    n = len(Z); kmax = int(np.sqrt(n)); P = np.zeros((n, size), np.int32)
    for i in range(size):
        k = int(rng.integers(2, kmax + 1))
        P[:, i] = KMeans(k, n_init=1, random_state=int(rng.integers(1 << 30))).fit_predict(Z)
    return P

def co_association(BP):
    cols = []
    for k in range(BP.shape[1]):
        _, lab = np.unique(BP[:, k], return_inverse=True)
        cols.append(np.eye(lab.max() + 1, dtype=np.float32)[lab])
    H = np.hstack(cols)
    return (H @ H.T) / BP.shape[1]

def knn_score(S, k):
    np.fill_diagonal(S, -np.inf)
    return 1 - (-np.partition(-S, k, axis=1)[:, :k]).mean(1)

# hybrid smoothing: Louvain community (small) or wired neighbours (large)
def neighbours(edges, nets):
    idx = {n: i for i, n in enumerate(nets)}; nb = [[] for _ in nets]
    for a, b in edges:
        if a in idx and b in idx: nb[idx[a]].append(idx[b]); nb[idx[b]].append(idx[a])
    return [np.array(sorted(set(v)), int) for v in nb]

def community_features(X, lab):
    Z = zscore(X); means = np.zeros((lab.max() + 1, Z.shape[1]))
    np.add.at(means, lab, Z); means /= np.bincount(lab)[:, None]
    return zscore(np.hstack([Z, Z - means[lab]]))

def louv_mean(z, lab):
    s = np.bincount(lab, weights=z); c = np.bincount(lab).astype(float)
    num, den = s[lab] - z, c[lab] - 1
    out = np.full_like(z, np.nan); has = den > 0
    out[has] = num[has] / den[has]
    return out

def hop_mean(z, nb):
    out = np.full_like(z, np.nan)
    for i, v in enumerate(nb):
        if len(v): out[i] = z[v].mean()
    return out

def smooth_raw(s, small, lab, nb):
    m = louv_mean(s, lab) if small else hop_mean(s, nb)
    return np.where(np.isnan(m), s, 0.5 * s + 0.5 * m)

# testability weight and automatic flag count
def hardness(D):
    cc = np.sqrt(D.CC0.to_numpy(float) ** 2 + D.CC1.to_numpy(float) ** 2)
    co = D.CO.to_numpy(float)
    return rankdata(np.log1p(cc) + np.log1p(co), 'max') / len(D)

def grow(S, seed, pool, th=CA_MAJ):
    fl = seed.copy()
    while True:
        new = fl | (pool & (S[:, fl] >= th).any(1))
        if new.sum() == fl.sum():
            return fl
        fl = new

def auto_flags(S, f):
    mx = f.max()
    if mx <= 0:
        return np.zeros(len(f), bool)
    return grow(S, f >= SEED_FRAC * mx, f >= POOL_FRAC * mx)

# evaluation (labels used only here)
def metrics(y, flag):
    tp = int((flag & (y == 1)).sum()); fp = int((flag & (y == 0)).sum())
    fn = int((~flag & (y == 1)).sum()); tn = int((~flag & (y == 0)).sum())
    tpr = tp / max(tp + fn, 1); tnr = tn / max(tn + fp, 1); p = tp / max(tp + fp, 1)
    f = 2 * p * tpr / (p + tpr) if p + tpr > 0 else 0.0
    return dict(TPR=tpr, TNR=tnr, P=p, F=f, A=(tp + tn) / len(y), TP=tp, FP=fp)

def rank_metrics(y, score):
    top = np.argsort(-score, kind='stable')[:int(y.sum())]
    return dict(AUC=roc_auc_score(y, score), AP=average_precision_score(y, score), Pn=y[top].mean())

In [5]:
# ---------------- DRA Louvain: directed, gate-weighted communities ----------------
GAMMA_GRID = [0.5, 1, 2, 4, 6, 8, 10, 15, 20]
MAX_FRAC = 0.30
SEM_W = {'xor_mux': 2.0, 'and_or': 1.5, 'ff': 1.5, 'not_buf': 1.0, 'other': 1.0}

def norm(s):
    return str(s).strip().lstrip('\\').strip()

def _one_level(G, m, partition, gamma, directed, rng):
    node2com = {u: i for i, u in enumerate(G.nodes())}
    inner = [{u} for u in G.nodes()]
    if directed:
        kin = dict(G.in_degree(weight='weight')); kout = dict(G.out_degree(weight='weight'))
        Sin, Sout = list(kin.values()), list(kout.values())
        nbrs = {}
        for u in G:
            nbrs[u] = defaultdict(float)
            for _, v, w in G.out_edges(u, data='weight'):
                if u != v: nbrs[u][v] += w
            for v, _, w in G.in_edges(u, data='weight'):
                if u != v: nbrs[u][v] += w
    else:
        deg = dict(G.degree(weight='weight')); Stot = list(deg.values())
        nbrs = {u: {v: d['weight'] for v, d in G[u].items() if v != u} for u in G}
    order = list(G.nodes); rng.shuffle(order)
    moves, improved, sweeps = 1, False, 0
    while moves > 0 and sweeps < 100:
        sweeps += 1; moves = 0
        for u in order:
            best_gain, best = 0, node2com[u]
            w2c = defaultdict(float)
            for v, w in nbrs[u].items(): w2c[node2com[v]] += w
            if directed:
                di, do = kin[u], kout[u]
                Sin[best] -= di; Sout[best] -= do
                remove = -w2c[best] / m + gamma * (do * Sin[best] + di * Sout[best]) / m ** 2
            else:
                d = deg[u]; Stot[best] -= d
                remove = -w2c[best] / m + gamma * (Stot[best] * d) / (2 * m ** 2)
            for c, w in w2c.items():
                if directed:
                    gain = remove + w / m - gamma * (do * Sin[c] + di * Sout[c]) / m ** 2
                else:
                    gain = remove + w / m - gamma * (Stot[c] * d) / (2 * m ** 2)
                if gain > best_gain + 1e-12:
                    best_gain, best = gain, c
            if directed: Sin[best] += di; Sout[best] += do
            else: Stot[best] += d
            if best != node2com[u]:
                com = G.nodes[u].get('nodes', {u})
                partition[node2com[u]].difference_update(com); inner[node2com[u]].remove(u)
                partition[best].update(com); inner[best].add(u)
                improved = True; moves += 1; node2com[u] = best
    return [p for p in partition if p], [p for p in inner if p], improved

def _aggregate(G, partition):
    H = G.__class__(); node2com = {}
    for i, part in enumerate(partition):
        nodes = set()
        for u in part:
            node2com[u] = i; nodes.update(G.nodes[u].get('nodes', {u}))
        H.add_node(i, nodes=nodes)
    for u, v, w in G.edges(data='weight'):
        a, b = node2com[u], node2com[v]
        H.add_edge(a, b, weight=w + (H[a][b]['weight'] if H.has_edge(a, b) else 0))
    return H

def louvain(G, gamma, seed=0, threshold=1e-7):
    rng = random.Random(seed)
    partition = [{u} for u in G.nodes()]
    if G.number_of_edges() == 0: return partition
    mod = modularity(G, partition, resolution=gamma, weight='weight')
    g = G.__class__(); g.add_nodes_from(G); g.add_weighted_edges_from(G.edges(data='weight', default=1))
    m = g.size(weight='weight'); directed = G.is_directed()
    partition, inner, improved = _one_level(g, m, partition, gamma, directed, rng)
    while True:
        best = [s.copy() for s in partition]
        new_mod = modularity(g, inner, resolution=gamma, weight='weight')
        if new_mod - mod <= threshold: return best
        mod = new_mod; g = _aggregate(g, inner)
        partition, inner, improved = _one_level(g, m, partition, gamma, directed, rng)
        if not improved: return [s.copy() for s in partition]

def sem_class(cell):
    c = str(cell).upper()
    if 'DFF' in c or 'LATCH' in c or c.startswith('LAT'): return 'ff'
    if 'XOR' in c or 'XNOR' in c or 'XNR' in c or 'MUX' in c or c.startswith('MX'): return 'xor_mux'
    if 'BUF' in c or c.startswith(('INV', 'NOT', 'I1S', 'IB1S', 'LSDN')): return 'not_buf'
    if c.startswith(('AND', 'OR', 'NAND', 'NOR', 'NND', 'NR', 'AO', 'OA', 'ISOLAND', 'ISOLOR')): return 'and_or'
    return 'other'

def build_graph(nets, edges, cells):
    idx = {n: i for i, n in enumerate(nets)}
    G = nx.DiGraph(); G.add_nodes_from(range(len(nets)))
    for a, b in edges:
        if a in idx and b in idx and a != b:
            w = SEM_W[sem_class(cells.get(norm(b), 'other'))]
            u, v = idx[a], idx[b]
            if G.has_edge(u, v): G[u][v]['weight'] += w
            else: G.add_edge(u, v, weight=w)
    return G

def dra_louvain(nets, edges, cells):
    G = build_graph(nets, edges, cells); n = G.number_of_nodes()
    rows, labs = [], {}
    for g in GAMMA_GRID:
        comms = louvain(G, g)
        lab = np.empty(n, int)
        for k, c in enumerate(comms): lab[list(c)] = k
        Q = modularity(G, comms, weight='weight', resolution=g)
        rows.append(dict(gamma=g, Q=Q, n_comm=len(comms), largest_frac=np.bincount(lab).max() / n)); labs[g] = lab
    tab = pd.DataFrame(rows); ok = tab[tab.largest_frac <= MAX_FRAC]
    g = float(ok.loc[ok.Q.idxmax(), 'gamma']) if len(ok) else float(tab.loc[tab.largest_frac.idxmin(), 'gamma'])
    return labs[g], g, tab
print('TW-CA-kNN code loaded.')

TW-CA-kNN code loaded.


## 4. Load the data

In [6]:
kw = dict(keep_default_na=False, na_values=[])   # some net names look like NA markers
S_ = pd.read_csv(os.path.join(DATA_DIR, 'structural_features.csv'), **kw)
T_ = pd.read_csv(os.path.join(DATA_DIR, 'testability_features.csv'), **kw)
M = S_.merge(T_.drop(columns='label'), on=['circuit', 'net']); M = M[M.net != '']
E_ = pd.read_csv(os.path.join(DATA_DIR, 'net_edges.csv'), **kw)
EDGES = {c: list(zip(g.src, g.dst)) for c, g in E_.groupby('circuit')}
Dc_ = pd.read_csv(os.path.join(DATA_DIR, 'net_driver_cells.csv'), **kw)
CELLS = {c: {norm(n): t for n, t in zip(g.net, g.driver_cell)} for c, g in Dc_.groupby('circuit')}
print(f"{M.circuit.nunique()} circuits, {len(M):,} nets, {int(M.label.sum())} Trojan nets")

15 circuits, 33,901 nets, 274 Trojan nets


## 5. Run TW-CA-kNN on every circuit

Each circuit runs `RUNS` times, each run drawing a different 20-member sample from the k-means pool.

In [7]:
def tw_caknn(c, D, edges, cells, runs):
    from threadpoolctl import threadpool_limits
    with threadpool_limits(1):
        t0 = time.time()
        y = D.label.to_numpy(int); nets = list(D.net); n = len(y); small = n < SMALL_N
        X = D[XGB22].to_numpy(float); h = hardness(D); lab, nb = None, None
        if small:
            lab = dra_louvain(nets, edges, cells)[0]
            P = kmeans_pool(community_features(X, lab), np.random.default_rng(SEED))
        else:
            nb = neighbours(edges, nets)
            P = kmeans_pool(zscore(X), np.random.default_rng(SEED))
        rows = []
        for r in range(runs):
            sel = np.random.default_rng(100 + r).choice(POOL, MES, replace=False)
            S = co_association(P[:, sel])
            s = knn_score(S, min(K_NN, n - 2)); np.fill_diagonal(S, 1.0)
            f = smooth_raw(s, small, lab, nb) * h ** BETA        # suspicion score
            flagged = auto_flags(S, f)
            rows.append(dict(circuit=c, run=r, **metrics(y, flagged), **rank_metrics(y, f)))
            del S
        return rows, dict(circuit=c, nets=n, trojan_nets=int(y.sum()),
                          branch='Louvain' if small else '1-hop', seconds=round(time.time() - t0, 1))

t0 = time.time()
circs = sorted(M.circuit.unique(), key=lambda c: -(M.circuit == c).sum())
OUT = Parallel(n_jobs=N_JOBS)(delayed(tw_caknn)(c, M[M.circuit == c].reset_index(drop=True),
                                                EDGES.get(c, []), CELLS.get(c, {}), RUNS) for c in circs)
RUN_RESULTS = pd.DataFrame([x for o in OUT for x in o[0]])
INFO = pd.DataFrame([o[1] for o in OUT]).set_index('circuit')
print(f'Done: {len(circs)} circuits x {RUNS} runs in {time.time() - t0:.0f}s')

Done: 15 circuits x 20 runs in 35s


## 6. Results

Per circuit: mean over runs. TP and FP are the mean number of Trojan nets found and false alarms. All other values are percentages. Recall equals TPR. P@n is precision when as many nets are flagged as there are Trojan nets.

In [8]:
order = sorted(INFO.index, key=lambda c: (not c.startswith('RS'), not c.startswith('s1423'), c))
PER = RUN_RESULTS.groupby('circuit')[['TPR', 'TNR', 'P', 'A', 'F', 'AUC', 'AP', 'Pn', 'TP', 'FP']].mean().loc[order]
PER[['TPR', 'TNR', 'P', 'A', 'F', 'AUC', 'AP', 'Pn']] *= 100
PER.insert(3, 'Recall', PER.TPR)
PER = PER.rename(columns={'P': 'Precision', 'A': 'Accuracy', 'Pn': 'P@n'})
PER = INFO.loc[order, ['nets', 'trojan_nets', 'branch']].join(PER)
display(PER.round(1))

SUMMARY = PER.drop(columns=['nets', 'trojan_nets', 'branch', 'TP', 'FP']).mean().to_frame('TW-CA-kNN (mean of 15 circuits)').T
SUMMARY['Trojans found'] = PER.TP.sum(); SUMMARY['false alarms'] = PER.FP.sum()
display(SUMMARY.round(2))

,nets,trojan_nets,branch,TPR,TNR,Precision,Recall,Accuracy,F,AUC,AP,P@n,TP,FP
circuit,,,,,,,,,,,,,,
RS232-T1000,283,11,Louvain,90.9,98.8,75.7,90.9,98.4,82.3,93.1,89.4,86.8,10.0,3.4
RS232-T1100,285,12,Louvain,91.7,99.0,81.3,91.7,98.7,85.7,94.0,91.8,90.4,11.0,2.8
RS232-T1200,288,13,Louvain,100.0,98.8,81.2,100.0,98.9,89.3,99.8,97.2,87.3,13.0,3.2
RS232-T1300,280,7,Louvain,100.0,99.4,85.5,100.0,99.4,91.1,100.0,99.9,99.3,7.0,1.6
RS232-T1400,284,12,Louvain,100.0,99.0,83.8,100.0,99.1,90.7,99.8,97.0,91.2,12.0,2.6
RS232-T1500,286,12,Louvain,91.7,98.9,80.0,91.7,98.6,85.0,93.7,91.7,90.8,11.0,3.0
RS232-T1600,280,7,Louvain,100.0,96.3,49.1,100.0,96.4,63.8,99.9,97.7,96.4,7.0,10.1
s1423_T401,576,37,Louvain,99.5,98.4,81.5,99.5,98.5,89.6,99.7,93.7,86.6,36.8,8.4
s1423_T402,556,17,Louvain,100.0,99.6,89.5,100.0,99.6,94.4,99.8,88.2,93.5,17.0,2.0


,TPR,TNR,Precision,Recall,Accuracy,F,AUC,AP,P@n,Trojans found,false alarms
TW-CA-kNN (mean of 15 circuits),89.98,99.22,87.17,89.98,99.15,85.09,97.68,92.7,90.14,252.75,37.1


## 7. Save the results

In [9]:
OUT_DIR = 'results_tw_caknn'; os.makedirs(OUT_DIR, exist_ok=True)
PER.round(4).to_csv(f'{OUT_DIR}/per_circuit.csv')
SUMMARY.round(4).to_csv(f'{OUT_DIR}/summary.csv')
RUN_RESULTS.to_csv(f'{OUT_DIR}/all_runs.csv', index=False)
print('Saved:', sorted(os.listdir(OUT_DIR)))
if IN_COLAB:
    import shutil; shutil.make_archive(OUT_DIR, 'zip', OUT_DIR)
    from google.colab import files; files.download(f'{OUT_DIR}.zip')

Saved: ['all_runs.csv', 'per_circuit.csv', 'summary.csv']
